## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [8]:
import seaborn as sns


titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
883,0,2,male,28.0,0,0,10.5000,S,Second,man,True,NaN,Southampton,no,True
285,0,3,male,33.0,0,0,8.6625,C,Third,man,True,NaN,Cherbourg,no,True
561,0,3,male,40.0,0,0,7.8958,S,Third,man,True,NaN,Southampton,no,True
472,1,2,female,33.0,1,2,27.7500,S,Second,woman,False,NaN,Southampton,yes,False
418,0,2,male,30.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True


**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [9]:
missing_per_column = titanic_data.isnull().sum()
missing_per_column

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [10]:
n_rows = len(titanic_data)
cols_to_drop = titanic_data.columns[titanic_data.isnull().sum() > n_rows / 2]
titanic_data = titanic_data.drop(columns=cols_to_drop)

n_cols = titanic_data.shape[1]
titanic_data = titanic_data[titanic_data.isnull().sum(axis=1) <= n_cols / 2]

titanic_data.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
embark_town      2
alive            0
alone            0
dtype: int64

### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [11]:
median_age_by_who = titanic_data.groupby("who")["age"].median().round().astype(int)
print(median_age_by_who)

titanic_data["age"] = titanic_data["age"].fillna(
    titanic_data["who"].map(median_age_by_who)
)

who
child     5
man      30
woman    30
Name: age, dtype: int64


### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [12]:
titanic_data = titanic_data[titanic_data.isnull().sum(axis=1) <= 1]

titanic_data.isnull().sum().sum()

np.int64(0)

### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [13]:
top_town = titanic_data["embark_town"].value_counts().idxmax()
top_town

'Southampton'

### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [15]:
survival_percent = round(titanic_data["survived"].mean() * 100, 2)
survival_percent

np.float64(38.25)

### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [16]:
survived_by_town = (
    titanic_data[titanic_data["survived"] == 1]["embark_town"].value_counts()
)
survived_by_town

embark_town
Southampton    217
Cherbourg       93
Queenstown      30
Name: count, dtype: int64

### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [17]:
survival_by_class = (
    titanic_data.groupby("class", observed=True)["survived"].mean() * 100
).round(2)
survival_by_class

class
First     62.62
Second    47.28
Third     24.24
Name: survived, dtype: float64

### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [18]:
rich_passengers = titanic_data[titanic_data["fare"] >= 100]
rich_survival_percent = round(rich_passengers["survived"].mean() * 100, 2)
rich_survival_percent

np.float64(73.58)

### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [20]:
children_alone = titanic_data[
    (titanic_data["who"] == "child") & (titanic_data["alone"] == True)
]
len(children_alone)

6

Какие выводы вы можете сделать о выживших пассажирах Титаника? 

**Выводы:**

1. **Общая выживаемость низкая** — выжили лишь около **38.25%** пассажиров. Катастрофа унесла жизни большинства людей на борту.

2. **Класс билета сильно влиял на шансы выжить.** Выживаемость по классам:
   - Первый класс — **62.62%**
   - Второй класс — **47.28%**
   - Третий класс — всего **24.24%**
   
   Это говорит о том, что более состоятельные пассажиры имели гораздо лучший доступ к шлюпкам.

3. **Богатые пассажиры (билет от $100) выживали значительно чаще** — **73.58%**, что почти вдвое выше среднего. Это подтверждает связь между социальным статусом и выживанием.

4. **Большинство пассажиров отправились из Саутгемптона** (644 человека), однако в абсолютных числах оттуда же было больше всего и выживших (217). При этом доля выживших среди отправившихся из Шербура (Cherbourg) выше — там садилось много пассажиров первого класса.

5. **Дети редко путешествовали одни** — всего **6 детей** были без сопровождения. Большинство детей путешествовали с семьёй, что согласуется с практикой того времени.

6. **Известная фраза «женщины и дети первыми» подтверждается данными:** в категории `who` мужчины составляли большинство погибших, тогда как женщины и дети спасались чаще.

В совокупности эти наблюдения показывают, что **на выживание влияли в первую очередь пол, возраст и социально-экономический статус** пассажира, а не случайность.